In [ ]:
# ============ CUADERNO 9 · PASO 1 · Entorno, manifiesto de archivos y procedencia ============
# Este cuaderno no toca ni regenera nada de lo anterior. Solo lee, mide y escribe archivos c9_*.
import os, sys, json, time, hashlib, glob
import numpy as np, pandas as pd, duckdb

BASE = r'C:\SERCOP_FINAL'
os.chdir(BASE)
R    = '03_resultados'
OUT  = f'{R}/c9_'
con  = duckdb.connect()
con.sql("SET preserve_insertion_order=false")
print('Carpeta de trabajo:', os.getcwd())

# ---------- 1.1 Manifiesto: huella y conteo de filas de cada archivo ----------
# Por que: hoy nada impide volver a sobrescribir un archivo intermedio. Ya paso una vez con
# base_maestra_v3. Con este manifiesto, si un archivo cambia, se nota al correr la celda de nuevo.
CARPETAS = ['00_json_crudo', '01_parquet', '01_datos', '03_resultados', '05_documentos']
EXT = ('.parquet', '.csv', '.xlsx')

def huella(ruta, trozo=8 * 1024 * 1024):
    """Huella rapida: tamano + primeros y ultimos 8 MB. Suficiente para detectar un cambio."""
    h = hashlib.blake2b(digest_size=12); n = os.path.getsize(ruta)
    h.update(str(n).encode())
    with open(ruta, 'rb') as f:
        h.update(f.read(trozo))
        if n > 2 * trozo:
            f.seek(-trozo, os.SEEK_END); h.update(f.read(trozo))
    return h.hexdigest()

filas, ilegibles = [], []
for carpeta in CARPETAS:
    if not os.path.isdir(carpeta): continue
    for ruta in sorted(glob.glob(os.path.join(carpeta, '**', '*'), recursive=True)):
        if not os.path.isfile(ruta) or not ruta.lower().endswith(EXT): continue
        nom = os.path.basename(ruta)
        if nom.startswith(('~$', '.')):            # archivos de bloqueo de Excel y temporales
            continue
        n = None
        if ruta.lower().endswith('.parquet'):
            try: n = con.sql(f"SELECT count(*) FROM read_parquet('{ruta.replace(chr(92),'/')}')").fetchone()[0]
            except Exception: n = None
        try:
            h = huella(ruta)
        except (PermissionError, OSError) as e:      # abierto en otro programa, o sin permiso
            ilegibles.append((ruta.replace(chr(92), '/'), type(e).__name__)); continue
        filas.append({'archivo': ruta.replace(chr(92), '/'),
                      'mb': round(os.path.getsize(ruta) / 1e6, 2),
                      'filas': n,
                      'modificado': time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(ruta))),
                      'huella': h})
man = pd.DataFrame(filas).sort_values('archivo')
if ilegibles:
    print(f'\nAVISO: {len(ilegibles)} archivo(s) no se pudieron leer (probablemente abiertos en Excel):')
    for a, e in ilegibles: print(f'  {a}  [{e}]')
    print('Cierralos y vuelve a correr la celda para que entren al manifiesto.')

# Comparacion con el manifiesto anterior, si existe
ANT = f'{OUT}manifiesto_archivos.csv'
if os.path.exists(ANT):
    viejo = pd.read_csv(ANT)[['archivo', 'huella', 'filas']].rename(
        columns={'huella': 'huella_antes', 'filas': 'filas_antes'})
    cmp = man.merge(viejo, on='archivo', how='outer', indicator=True)
    cambio = cmp[(cmp._merge == 'both') & (cmp.huella != cmp.huella_antes)]
    nuevos = cmp[cmp._merge == 'left_only']; idos = cmp[cmp._merge == 'right_only']
    print(f'\n--- Contra el manifiesto anterior ---')
    print(f'  archivos cambiados: {len(cambio)} | nuevos: {len(nuevos)} | desaparecidos: {len(idos)}')
    if len(cambio):
        print('  ATENCION, estos archivos cambiaron:')
        print(cambio[['archivo', 'filas_antes', 'filas']].to_string(index=False))
    if len(idos):
        print('  ATENCION, estos archivos ya no estan:')
        print(idos[['archivo', 'filas_antes']].to_string(index=False))
else:
    print('\nNo hay manifiesto anterior: este es el punto de partida.')

man.to_csv(ANT, index=False)
print(f'\n{len(man)} archivos registrados | {man.mb.sum() / 1000:.2f} GB en total')
print(f'  -> {ANT}')
print(man[man.mb > 5][['archivo', 'mb', 'filas', 'modificado']].to_string(index=False))

# ---------- 1.2 Procedencia ----------
# Si nadie anoto la fecha de descarga, se reconstruye de los archivos y SE DECLARA ASI.
# En el documento final va como "fecha de corte reconstruida", no como dato verificado.
crudos = man[man.archivo.str.contains('01_datos')]
PROCEDENCIA = {
    'fuente': 'Portal de datos abiertos de contratacion publica del Ecuador (SERCOP), formato OCDS',
    'direccion': 'https://datosabiertos.compraspublicas.gob.ec/',          # confirmar la direccion exacta
    'fecha_descarga_declarada': None,            # <- PONER LA FECHA SI ALGUIEN LA RECUERDA (AAAA-MM-DD)
    'fecha_descarga_reconstruida': (crudos.modificado.min()[:10] if len(crudos) else None),
    'archivo_mas_antiguo': (crudos.archivo.iloc[crudos.modificado.argmin()] if len(crudos) else None),
    'anio_minimo_en_datos': None, 'anio_maximo_en_datos': None,   # se llenan en el paso 2
    'nota': 'Si fecha_descarga_declarada queda en None, el trabajo reporta la reconstruida y lo dice.',
}
print('\n--- Procedencia ---')
for k, v in PROCEDENCIA.items(): print(f'  {k:32s} {v}')
with open(f'{OUT}procedencia.json', 'w', encoding='utf-8') as f:
    json.dump(PROCEDENCIA, f, ensure_ascii=False, indent=1)
print(f'  -> {OUT}procedencia.json')
print('\nOJO: rellena a mano fecha_descarga_declarada y direccion antes de seguir.')

In [ ]:
# ============ CUADERNO 9 · PASO 2 · Perfilado por columna y reglas de validacion ============
# Requiere el Paso 1 corrido (usa con, OUT, R).
BM4   = f'{R}/base_maestra_v4.parquet'
PANEL = f'{R}/c8_panel.parquet'
for ruta, nombre in [(BM4, 'bm4'), (PANEL, 'panel')]:
    con.sql(f"CREATE OR REPLACE VIEW {nombre} AS SELECT * FROM read_parquet('{ruta}')")
print('bm4  :', con.sql('SELECT count(*) FROM bm4').fetchone()[0], 'filas x',
      len(con.sql('DESCRIBE bm4').fetchall()), 'columnas')
print('panel:', con.sql('SELECT count(*) FROM panel').fetchone()[0], 'filas x',
      len(con.sql('DESCRIBE panel').fetchall()), 'columnas')

# ---------- 2.1 Perfilado: una fila por columna ----------
# SUMMARIZE de DuckDB lee las estadisticas del parquet, asi que es rapido aun con millones de filas.
perfiles = []
for nombre in ['bm4', 'panel']:
    s = con.sql(f'SUMMARIZE {nombre}').df()
    s.insert(0, 'tabla', nombre)
    perfiles.append(s)
perfil = pd.concat(perfiles, ignore_index=True)
perfil['pct_nulos'] = perfil.null_percentage.astype(float).round(2)
perfil['constante'] = perfil.approx_unique.fillna(0).astype(int) <= 1
perfil.to_csv(f'{OUT}perfil_columnas.csv', index=False)
print(f'\n  -> {OUT}perfil_columnas.csv ({len(perfil)} columnas perfiladas)')

print('\n--- Columnas vacias o casi vacias (>= 95 % nulos) ---')
vac = perfil[perfil.pct_nulos >= 95][['tabla', 'column_name', 'pct_nulos']]
print(vac.to_string(index=False) if len(vac) else '  ninguna')
print('\n--- Columnas constantes (no aportan nada y deben salir o justificarse) ---')
cte = perfil[perfil.constante][['tabla', 'column_name', 'min', 'count']]
print(cte.to_string(index=False) if len(cte) else '  ninguna')
print('\n--- Las 15 columnas con mas nulos de la base maestra ---')
print(perfil[perfil.tabla == 'bm4'].nlargest(15, 'pct_nulos')[
      ['column_name', 'column_type', 'pct_nulos', 'approx_unique']].to_string(index=False))

# ---------- 2.2 Reglas de validacion, cada una con su conteo ----------
# Se declaran como expresiones SQL. Una regla que no se puede evaluar por falta de columna
# se informa como omitida: eso tambien es informacion.
cols_bm4 = {r[0] for r in con.sql('DESCRIBE bm4').fetchall()}
C_MONTO  = next((c for c in ('monto_tender', 'monto_final', 'monto_adjudicado') if c in cols_bm4), None)
C_FECHA  = next((c for c in ('fecha_release', 'fecha_proceso', 'fecha') if c in cols_bm4), None)
ANIO_MAX = con.sql('SELECT max(anio_proceso) FROM bm4').fetchone()[0] if 'anio_proceso' in cols_bm4 else 2026
print(f'\nColumnas detectadas -> monto: {C_MONTO} | fecha: {C_FECHA} | anio maximo: {ANIO_MAX}')

REGLAS = [
 ('R01 RUC del proveedor presente',      {'proveedor_ruc'}, "TRIM(COALESCE(proveedor_ruc,'')) = ''"),
 ('R02 RUC de 13 caracteres',            {'proveedor_ruc'}, "TRIM(COALESCE(proveedor_ruc,'')) <> '' AND length(TRIM(proveedor_ruc)) <> 13"),
 ('R03 RUC solo digitos',                {'proveedor_ruc'}, "TRIM(COALESCE(proveedor_ruc,'')) <> '' AND NOT regexp_matches(TRIM(proveedor_ruc), '^[0-9]+$')"),
 ('R04 RUC termina en 001',              {'proveedor_ruc'}, "length(TRIM(COALESCE(proveedor_ruc,''))) = 13 AND right(TRIM(proveedor_ruc),3) <> '001'"),
 ('R05 provincia del RUC entre 01 y 24', {'proveedor_ruc'}, "length(TRIM(COALESCE(proveedor_ruc,''))) = 13 AND NOT (try_cast(left(TRIM(proveedor_ruc),2) AS INT) BETWEEN 1 AND 24)"),
 ('R06 tercer digito del RUC valido',    {'proveedor_ruc'}, "length(TRIM(COALESCE(proveedor_ruc,''))) = 13 AND substr(TRIM(proveedor_ruc),3,1) NOT IN ('0','1','2','3','4','5','6','9')"),
 ('R07 RUC de la entidad presente',      {'entidad_ruc'},   "TRIM(COALESCE(entidad_ruc,'')) = ''"),
 ('R08 identificador del proceso presente', {'ocid'},       "ocid IS NULL OR TRIM(ocid) = ''"),
 ('R09 anio del proceso presente',       {'anio_proceso'},  "anio_proceso IS NULL"),
 ('R10 anio entre 2015 y el maximo',     {'anio_proceso'},  f"anio_proceso IS NOT NULL AND (anio_proceso < 2015 OR anio_proceso > {ANIO_MAX})"),
]
if C_MONTO:
    REGLAS += [(f'R11 {C_MONTO} no negativo', {C_MONTO}, f"{C_MONTO} < 0"),
               (f'R12 {C_MONTO} distinto de cero', {C_MONTO}, f"{C_MONTO} = 0")]
if C_FECHA:
    REGLAS += [(f'R13 {C_FECHA} presente', {C_FECHA}, f"{C_FECHA} IS NULL"),
               (f'R14 {C_FECHA} coherente con el anio', {C_FECHA, 'anio_proceso'},
                f"{C_FECHA} IS NOT NULL AND anio_proceso IS NOT NULL AND year(try_cast({C_FECHA} AS DATE)) <> anio_proceso")]
if 'n_oferentes' in cols_bm4:
    REGLAS += [('R15 n_oferentes no negativo', {'n_oferentes'}, "n_oferentes < 0"),
               ('R16 n_oferentes igual a cero',  {'n_oferentes'}, "n_oferentes = 0")]

total = con.sql('SELECT count(*) FROM bm4').fetchone()[0]
filas_r, omitidas = [], []
for nombre, necesita, expr in REGLAS:
    if not necesita <= cols_bm4:
        omitidas.append((nombre, sorted(necesita - cols_bm4))); continue
    try:
        n = con.sql(f'SELECT count(*) FROM bm4 WHERE {expr}').fetchone()[0]
        filas_r.append({'regla': nombre, 'violaciones': n, 'pct': round(n / total * 100, 4)})
    except Exception as e:
        omitidas.append((nombre, [f'error: {str(e)[:70]}']))
val = pd.DataFrame(filas_r).sort_values('violaciones', ascending=False)
print(f'\n--- Reglas de validacion sobre {total:,} registros ---')
print(val.to_string(index=False))
if omitidas:
    print('\nReglas omitidas (y por que):')
    for n, m in omitidas: print(f'  {n}: falta {m}')
val.to_csv(f'{OUT}reglas_validacion.csv', index=False)

# ---------- 2.3 Grano y duplicados ----------
print('\n--- Grano de la tabla ---')
if 'ocid' in cols_bm4:
    g = con.sql("""SELECT count(*) AS filas, count(DISTINCT ocid) AS ocid_distintos,
                          round(count(*)*1.0/count(DISTINCT ocid),2) AS filas_por_ocid FROM bm4""").df()
    print(g.to_string(index=False))
    if 'proveedor_ruc' in cols_bm4:
        d = con.sql("""SELECT count(*) AS pares_repetidos FROM (
                         SELECT ocid, proveedor_ruc, count(*) c FROM bm4
                         GROUP BY 1,2 HAVING c > 1)""").fetchone()[0]
        print(f'  pares (ocid, proveedor_ruc) que aparecen mas de una vez: {d:,}')
        print('  Si este numero no es cero, hay que decidir y documentar que significa una fila:')
        print('  un proceso, un lote, un item o una adjudicacion.')
print(f'\n  -> {OUT}reglas_validacion.csv')

In [ ]:
# ============ CUADERNO 9 · PASO 3 · Procedencia, cadena de filas y los tres casos raros ============
# Requiere los pasos 1 y 2 corridos (usa con, OUT, R, bm4, panel).

# ---------- 3.1 Buscar los datos crudos ----------
# El manifiesto no encontro ni un archivo de la descarga original: todo empieza en
# base_modelado_v1. Sin el crudo no se puede decir cuantos registros se descargaron.
# Esta celda lo busca en todo C:\SERCOP_FINAL antes de darlo por perdido.
import glob, os, time
CRUDO_EXT = ('.json', '.jsonl', '.zip', '.csv', '.parquet', '.gz', '.7z', '.rar')
candidatos = []
for ruta in glob.glob(os.path.join('.', '**', '*'), recursive=True):
    if not os.path.isfile(ruta): continue
    nom = os.path.basename(ruta).lower()
    if nom.startswith(('~$', '.')) or not nom.endswith(CRUDO_EXT): continue
    if ruta.replace('\\', '/').startswith('./03_resultados'): continue      # ya sabemos que hay ahi
    candidatos.append((ruta.replace('\\', '/'), round(os.path.getsize(ruta) / 1e6, 1),
                       time.strftime('%Y-%m-%d', time.localtime(os.path.getmtime(ruta)))))
candidatos.sort(key=lambda t: -t[1])
print('--- Archivos fuera de 03_resultados, de mayor a menor ---')
for a, mb, f in candidatos[:25]: print(f'  {mb:>8.1f} MB  {f}  {a}')
if not candidatos: print('  ninguno')
print('\nSi aqui no aparece la descarga original (OCDS en json o zip), NO esta en la carpeta')
print('del proyecto. Hay que buscarla en Descargas, en un disco externo o volver a descargarla.')

# ---------- 3.2 Procedencia corregida ----------
rango = con.sql('SELECT min(anio_proceso) a, max(anio_proceso) b, count(*) n FROM bm4').df().iloc[0]
man = pd.read_csv(f'{OUT}manifiesto_archivos.csv')
primero = man.sort_values('modificado').iloc[0]
PROCEDENCIA = {
 'fuente': 'Portal de datos abiertos de contratacion publica del Ecuador (SERCOP), formato OCDS',
 'direccion': 'https://datosabiertos.compraspublicas.gob.ec/',        # confirmar
 'fecha_descarga_declarada': None,            # <- PONER LA FECHA SI ALGUIEN LA RECUERDA
 'artefacto_mas_antiguo': primero.archivo,
 'fecha_artefacto_mas_antiguo': primero.modificado,
 'crudo_localizado': bool(candidatos),
 'anio_minimo_en_datos': int(rango.a), 'anio_maximo_en_datos': int(rango.b),
 'registros_en_base_maestra': int(rango.n),
 'nota': ('La descarga original no esta en la carpeta del proyecto. La cadena documentada '
          'empieza en el artefacto mas antiguo y eso se declara como limite de reproducibilidad.'),
}
print('\n--- Procedencia ---')
for k, v in PROCEDENCIA.items(): print(f'  {k:30s} {v}')
with open(f'{OUT}procedencia.json', 'w', encoding='utf-8') as f:
    json.dump(PROCEDENCIA, f, ensure_ascii=False, indent=1)

# ---------- 3.3 La cadena de filas, reconstruida del manifiesto ----------
# Donde el conteo cambia, hubo un filtro. Cada salto hay que poder explicarlo.
ch = man[man.filas.notna() & man.archivo.str.endswith('.parquet')].copy()
ch = ch.sort_values('modificado')[['modificado', 'archivo', 'filas']]
ch['filas'] = ch.filas.astype(int)
ch['salto'] = ch.filas.diff().fillna(0).astype(int)
print('\n--- Cadena de artefactos en orden de creacion ---')
print(ch.to_string(index=False))
saltos = ch[(ch.salto != 0) & (ch.salto.abs() > 100)]
print('\n--- Saltos de mas de 100 filas (cada uno necesita una explicacion escrita) ---')
print(saltos.to_string(index=False) if len(saltos) else '  ninguno')
ch.to_csv(f'{OUT}cadena_filas.csv', index=False)

# ---------- 3.4 Los tres casos raros que encontro el paso 2 ----------
print('\n' + '='*70 + '\n3.4  Los casos que hay que mirar de frente\n' + '='*70)
cols_bm4 = {r[0] for r in con.sql('DESCRIBE bm4').fetchall()}

print('\n--- (a) 46.370 registros sin RUC de proveedor (1,69 %) ---')
print(con.sql("""
SELECT CASE WHEN proveedor_ruc IS NULL THEN 'nulo' ELSE 'cadena vacia' END AS tipo,
       count(*) AS registros, round(sum(monto_tender)/1e6, 1) AS monto_millones,
       min(anio_proceso) AS anio_min, max(anio_proceso) AS anio_max
FROM bm4 WHERE TRIM(COALESCE(proveedor_ruc,'')) = '' GROUP BY 1 ORDER BY 2 DESC""").df().to_string(index=False))
print('Estos 46.370 ya estan excluidos del panel (bmv los filtra). Lo que falta es declararlo')
print('en la cadena de filtros: 2.736.387 menos 46.370 da los 2.690.017 del panel de procesos.')

print('\n--- (b) 7 registros con provincia de RUC fuera de 01-24 ---')
print(con.sql("""
SELECT left(TRIM(proveedor_ruc),2) AS provincia, count(*) AS registros,
       min(TRIM(proveedor_ruc)) AS ejemplo, min(anio_proceso) AS anio
FROM bm4 WHERE length(TRIM(COALESCE(proveedor_ruc,''))) = 13
  AND NOT (try_cast(left(TRIM(proveedor_ruc),2) AS INT) BETWEEN 1 AND 24)
GROUP BY 1 ORDER BY 2 DESC""").df().to_string(index=False))
print('El codigo 30 se usa para contribuyentes sin cedula ecuatoriana (extranjeros).')
print('Si es eso, la regla se corrige para aceptarlo y se documenta. Si no, son datos malos.')

print('\n--- (c) 16 registros con monto cero ---')
print(con.sql("""
SELECT anio_proceso, tipo_contratacion, count(*) AS registros
FROM bm4 WHERE monto_tender = 0 GROUP BY 1,2 ORDER BY 3 DESC LIMIT 10""").df().to_string(index=False))

print('\n--- (d) Columnas constantes: nunca cambian de valor ---')
for t, c in [('bm4','entidad_es_fantasma'), ('bm4','grupo_nominal'),
             ('panel','sup_asociacion_o_consorc')]:
    if c in ({r[0] for r in con.sql(f'DESCRIBE {t}').fetchall()}):
        v = con.sql(f'SELECT DISTINCT {c} FROM {t}').df()
        print(f'  {t}.{c}: valores distintos = {v.iloc[:,0].tolist()}')
print('Las tres salen del modelo. Pero `grupo_nominal` en false para los 2,7 millones significa')
print('que esa variable NUNCA se activo: hay que revisar si esta mal construida o si el fenomeno')
print('simplemente no existe en los datos. Son dos conclusiones muy distintas.')

print('\n--- (e) Las 7 columnas de la Superintendencia de la ENTIDAD estan al 98,7 % vacias ---')
print(con.sql("""
SELECT count(*) AS total,
       sum(CASE WHEN entidad_superintendencia_tipo IS NOT NULL THEN 1 ELSE 0 END) AS con_dato
FROM bm4""").df().to_string(index=False) if 'entidad_superintendencia_tipo' in cols_bm4 else '  columna ausente')
print('Tiene sentido: las entidades contratantes son publicas y no estan en el directorio de')
print('companias. Son 7 columnas que entran al modelo como ruido. Explica por que quitar la')
print('familia ent_ en la ablacion solo costaba 0,009.')
print(f'\n  -> {OUT}cadena_filas.csv, {OUT}procedencia.json')

In [ ]:
# ============ CUADERNO 9 · PASO 4b · La cadena cerrada (corrige dos errores del 4) ============
# Error 1: el total del crudo excluia los DOS archivos de 2024 en vez de solo el no reparado.
#          El total correcto es 5.070.819 releases, no 4.682.010.
# Error 2: n_filas() consultaba DuckDB y al primer fallo la transaccion quedaba abortada, asi
#          que todo lo siguiente devolvia NaN. Ahora los conteos se leen del manifiesto, que ya
#          los tiene, y no se vuelve a tocar la base.
import os, json
import numpy as np, pandas as pd

man    = pd.read_csv(f'{OUT}manifiesto_archivos.csv')
crudo  = pd.read_csv(f'{OUT}crudo_releases.csv')
N = lambda sufijo: (lambda s: int(s.iloc[0]) if len(s) and pd.notna(s.iloc[0]) else None)(
        man.loc[man.archivo.str.endswith(sufijo), 'filas'])

# ---------- 4.2b Total del crudo, bien contado ----------
crudo['anio'] = crudo.archivo.str.extract(r'(\d{4})').astype(int)
crudo['reparado'] = crudo.archivo.str.contains('reparado')
# Por cada anio se usa el reparado si existe; si no, el normal.
anios_con_rep = set(crudo.loc[crudo.reparado, 'anio'])
crudo['cuenta'] = crudo.apply(lambda r: r.reparado or (r.anio not in anios_con_rep), axis=1)
TOTAL_CRUDO = int(crudo.loc[crudo.cuenta, 'releases'].sum())
print('--- Releases del crudo, por anio ---')
t = crudo[crudo.cuenta].sort_values('anio')[['anio', 'releases']].reset_index(drop=True)
# procesos unicos por anio, del primer parquet de cada anio
t['procesos'] = t.anio.map(lambda a: next((int(v) for s, v in
    zip(man.archivo, man.filas) if f'01_parquet/releases_{a}' in s and pd.notna(v)), np.nan))
t['releases_por_proceso'] = (t.releases / t.procesos).round(2)
print(t.to_string(index=False))
print(f'\nTotal de releases (usando 2024 reparado): {TOTAL_CRUDO:,}')
print(f'Total de procesos unicos por anio:        {int(t.procesos.sum()):,}')

# ---------- 4.3b La cadena, con cada eslabon verificado ----------
N_CONSOL   = N('01_parquet/sercop_consolidado.parquet')
N_PLANEA   = N('procesos_solo_planeacion.parquet')
N_MOD1     = N('base_modelado_v1.parquet')
N_SINMONTO = N('procesos_sin_monto.parquet')
N_BM4      = N('base_maestra_v4.parquet')
N_ETIQ     = N('c8_etiqueta.parquet')
N_PANEL    = N('c8_panel.parquet')
N_SINRUC   = N_BM4 - N_ETIQ if (N_BM4 and N_ETIQ) else None

cad = [
 ('0', 'Descarga OCDS del portal del SERCOP, 12 anios de releases', TOTAL_CRUDO,
  '00_json_crudo/releases_*.json', 'corte del 5 de septiembre de 2026'),
 ('1', 'Un release es un evento, no un proceso: se consolidan por ocid', N_CONSOL,
  '01_parquet/sercop_consolidado.parquet', f'{TOTAL_CRUDO/N_CONSOL:.2f} releases por proceso'),
 ('2', 'Se excluyen los procesos que solo llegaron a planeacion', -(N_PLANEA or 0),
  'procesos_solo_planeacion.parquet', 'no hay adjudicacion que analizar'),
 ('3', 'Base de modelado inicial', N_MOD1, 'base_modelado_v1.parquet', ''),
 ('4', 'Se excluyen los procesos sin monto', -(N_SINMONTO or 0),
  'procesos_sin_monto.parquet', 'sin monto no hay nada que medir'),
 ('5', 'Base maestra de procesos', N_BM4, 'base_maestra_v4.parquet', ''),
 ('6', 'Se excluyen los registros sin RUC de proveedor', -(N_SINRUC or 0),
  'filtro de la vista bmv', '42.294 cadena vacia y 4.076 nulo'),
 ('7', 'Universo de procesos con proveedor identificado', N_ETIQ, 'c8_etiqueta.parquet', ''),
 ('8', 'Agregacion a proveedor-anio', N_PANEL, 'c8_panel.parquet', 'unidad de analisis del modelo'),
]
cadena = pd.DataFrame(cad, columns=['paso', 'que_pasa', 'filas', 'archivo', 'nota'])
print('\n' + '='*72 + '\n4.3b  La cadena de filtros, de la descarga al panel\n' + '='*72)
print(cadena.to_string(index=False))

print('\n--- Verificacion, eslabon por eslabon ---')
ok_todo = True
def chk(nombre, calculado, esperado):
    global ok_todo
    ok = calculado == esperado
    ok_todo &= ok
    print(f'  {nombre:50s} {"CUADRA   " if ok else "NO CUADRA"} {calculado:>10,} vs {esperado:>10,}')
chk('suma de procesos por anio = consolidado', int(t.procesos.sum()), N_CONSOL)
chk('consolidado - solo planeacion = modelado_v1', N_CONSOL - N_PLANEA, N_MOD1)
chk('modelado_v1 - sin monto = base maestra',     N_MOD1 - N_SINMONTO, N_BM4)
chk('base maestra - sin RUC = universo de procesos', N_BM4 - N_SINRUC, N_ETIQ)
print(f'\n  {"TODA LA CADENA CUADRA" if ok_todo else "HAY UN ESLABON QUE NO CUADRA: revisarlo antes de seguir"}')
cadena.to_csv(f'{OUT}cadena_filtros.csv', index=False)
t.to_csv(f'{OUT}crudo_por_anio.csv', index=False)

# ---------- 4.3c Dos cosas que hay que declarar del propio conteo ----------
print('\n--- Lo que el conteo por anio deja ver ---')
pico = t.loc[t.releases.idxmax()]; caida = t[t.anio.isin([2019, 2020])]
if len(caida) == 2:
    a, b = caida.releases.tolist()
    print(f'  2019 -> 2020: {a:,} a {b:,} releases ({(b/a-1)*100:.0f} %). Es la pandemia y merece')
    print('  tratamiento aparte en el capitulo de resultados.')
ult = t.iloc[-1]
print(f'  {int(ult.anio)} esta INCOMPLETO: {int(ult.releases):,} releases hasta la fecha de corte.')
print('  El percentil dentro del anio no se afecta, pero cualquier comparacion de niveles')
print('  entre anios tiene que excluir o anualizar ese anio. Hay que declararlo.')

# ---------- 4.4b Las dos variables que nunca se activaron ----------
print('\n' + '='*72 + '\n4.4b  grupo_nominal y sup_asociacion_o_consorc\n' + '='*72)
try: con.rollback()                       # por si quedo una transaccion abortada
except Exception: pass
con.sql(f"CREATE OR REPLACE VIEW bm4 AS SELECT * FROM read_parquet('{R}/base_maestra_v4.parquet')")
cols_bm4 = {r[0] for r in con.sql('DESCRIBE bm4').fetchall()}
if 'grupo_nominal' in cols_bm4:
    print(con.sql('SELECT grupo_nominal, count(*) AS registros FROM bm4 GROUP BY 1 ORDER BY 2 DESC')
          .df().to_string(index=False))
    print('Solo false y nulo = la variable nunca se activo en 2,7 millones de registros.')
    print('Hay que abrir el cuaderno donde nacio y ver la condicion: o esta mal escrita, o el')
    print('fenomeno no existe en los datos. Son dos conclusiones distintas y una es un error nuestro.')
d = pd.read_parquet(f'{R}/directorio_companias.parquet')
col_tipo = next((c for c in d.columns if c.strip().upper() == 'TIPO'), None)
if col_tipo:
    print(f'\n--- Valores reales del campo {col_tipo} en el directorio ---')
    print(d[col_tipo].value_counts(dropna=False).to_string())
    print('\nSi ninguna categoria dice asociacion ni consorcio, el indicador se construyo contra')
    print('una etiqueta inexistente. Si existe pero con pocos casos, hay que ver por que ninguno')
    print('de esos RUC aparece en el panel.')
print(f'\n  -> {OUT}cadena_filtros.csv, {OUT}crudo_por_anio.csv')

In [ ]:
# ============ CUADERNO 9 · PASO 5b · Taxonomia cerrada + dos hallazgos ============
# Los 68.976 "sin clasificar" NO son basura: son las diez categorias del Regimen Especial de
# la LOSNCP, donde la ley permite un solo proveedor. Su 98,6 % de oferente unico es por diseno,
# no una anomalia. Con estas reglas la clasificacion queda al 100 %.
# Ademas: la suma de monto de Subasta Inversa salio NaN, asi que hay NaN en monto_tender que
# las reglas del Paso 2 no detectaron (NaN < 0 y NaN = 0 son ambos falsos). Aqui se cuentan.

FAMILIA = """
 CASE
   WHEN tipo_contratacion IS NULL THEN 'Sin tipo declarado'
  WHEN t LIKE '%CATALOGO ELECTRONICO%'            THEN 'Catalogo electronico'
  WHEN t LIKE '%SUBASTA INVERSA%'                 THEN 'Subasta inversa electronica'
  WHEN t LIKE '%LICITACION%'                      THEN 'Licitacion'
  WHEN t LIKE '%COTIZACION%'                      THEN 'Cotizacion'
  WHEN t LIKE '%CONCURSO PUBLICO%'                THEN 'Concurso publico'
  WHEN t LIKE '%MENOR CUANTIA%'                   THEN 'Menor cuantia'
  WHEN t LIKE '%LISTA CORTA%'                     THEN 'Lista corta'
  WHEN t LIKE '%INFIMA CUANTIA%'                  THEN 'Infima cuantia'
  WHEN t LIKE '%CONTRATACION DIRECTA%'            THEN 'Contratacion directa'
  WHEN t LIKE '%FERIA%'                           THEN 'Ferias inclusivas'
  WHEN t LIKE '%EMERGENCIA%'                      THEN 'Emergencia'
  -- Regimen Especial (LOSNCP art. 2): la ley admite un solo proveedor
  WHEN t LIKE '%BIENES Y SERVICIOS UNICOS%'       THEN 'Regimen especial'
  WHEN t LIKE '%ENTIDADES PUBLICAS%'              THEN 'Regimen especial'
  WHEN t LIKE '%OBRA ARTISTICA%'                  THEN 'Regimen especial'
  WHEN t LIKE '%REPUESTOS O ACCESORIOS%'          THEN 'Regimen especial'
  WHEN t LIKE '%COMUNICACION SOCIAL%'             THEN 'Regimen especial'
  WHEN t LIKE '%SEGUROS%'                         THEN 'Regimen especial'
  WHEN t LIKE '%PATROCINIO JURIDICO%'             THEN 'Regimen especial'
  WHEN t LIKE '%TRANSPORTE DE CORREO%'            THEN 'Regimen especial'
  WHEN t LIKE '%EMPRESAS PUBLICAS INTERNACIONALES%' THEN 'Regimen especial'
  WHEN t LIKE '%FARMACO%' OR t LIKE '%MEDICAMENTO%' THEN 'Regimen especial'
  WHEN t LIKE '%SEGURIDAD INTERNA%' OR t LIKE '%SECTORES ESTRATEGICOS%' THEN 'Regimen especial'
  WHEN t LIKE '%REGIMEN ESPECIAL%'                THEN 'Regimen especial'
  WHEN t LIKE '%CONSULTORIA%'                     THEN 'Consultoria'
  WHEN t LIKE '%SUBASTA%'                         THEN 'Otras subastas'
  ELSE 'Sin clasificar' END"""
CLASE = """
 CASE
  WHEN fam = 'Catalogo electronico' THEN '1 compra por catalogo'
  WHEN fam IN ('Subasta inversa electronica','Licitacion','Cotizacion','Concurso publico')
                                    THEN '2 abierto competitivo'
  WHEN fam IN ('Menor cuantia','Lista corta','Consultoria','Ferias inclusivas','Otras subastas')
                                    THEN '3 selectivo o por invitacion'
  WHEN fam = 'Regimen especial'     THEN '4 regimen especial (la ley admite proveedor unico)'
  WHEN fam IN ('Contratacion directa','Infima cuantia','Emergencia') THEN '5 directo'
  ELSE '9 sin clasificar' END"""
con.sql(f"""
CREATE OR REPLACE VIEW bm4t AS
WITH a AS (SELECT *, upper(strip_accents(tipo_contratacion)) AS t FROM bm4),
     b AS (SELECT *, {FAMILIA} AS fam FROM a)
SELECT * EXCLUDE (t), {CLASE} AS clase,
       (fam = 'Catalogo electronico') AS es_catalogo,
       ({CLASE} = '2 abierto competitivo') AS universo_competencia
FROM b""")

print('='*74 + '\n5.2b  Taxonomia por familia\n' + '='*74)
res = con.sql("""
SELECT clase, fam AS familia, count(*) AS procesos,
       round(sum(CASE WHEN isnan(monto_tender) THEN NULL ELSE monto_tender END)/1e6, 1) AS monto_millones,
       round(avg(CASE WHEN proveedor_unico THEN 1.0 ELSE 0.0 END)*100, 1) AS pct_of_unico,
       round(avg(CASE WHEN n_oferentes IS NOT NULL THEN 1.0 ELSE 0.0 END)*100, 1) AS pct_con_oferentes,
       sum(CASE WHEN isnan(monto_tender) THEN 1 ELSE 0 END) AS montos_nan
FROM bm4t GROUP BY 1,2 ORDER BY 1, 3 DESC""").df()
print(res.to_string(index=False))
res.to_csv(f'{OUT}taxonomia_procedimiento.csv', index=False)

sinc = con.sql("SELECT count(*) FROM bm4t WHERE clase = '9 sin clasificar'").fetchone()[0]
print(f'\nSin clasificar: {sinc:,}')
if sinc:
    print(con.sql("""SELECT substr(tipo_contratacion,1,60) AS valor, count(*) AS procesos
      FROM bm4t WHERE clase = '9 sin clasificar' GROUP BY 1 ORDER BY 2 DESC LIMIT 10""").df().to_string(index=False))
    print('  Agrega una regla por cada uno arriba y vuelve a correr la celda.')
else:
    print('  La clasificacion esta completa. Ninguna fila queda fuera.')

# ---------- El hallazgo de los nulos ----------
print('\n' + '='*74 + '\n5.6  De donde salen en realidad los nulos de n_oferentes\n' + '='*74)
nul = con.sql("""
SELECT clase, count(*) AS procesos,
       sum(CASE WHEN n_oferentes IS NULL THEN 1 ELSE 0 END) AS sin_dato,
       round(avg(CASE WHEN n_oferentes IS NULL THEN 1.0 ELSE 0.0 END)*100, 2) AS pct_sin_dato
FROM bm4t GROUP BY 1 ORDER BY 1""").df()
print(nul.to_string(index=False))
tot = int(nul.procesos.sum()); sd = int(nul.sin_dato.sum())
print(f'\n  Nulos de n_oferentes en toda la base: {sd:,} de {tot:,} ({sd/tot*100:.2f} %)')
cat = nul[nul.clase.str.contains('catalogo')]
if len(cat) and sd > 0:
    print(f'  De esos, del catalogo electronico: {int(cat.sin_dato.iloc[0]):,} '
          f'({int(cat.sin_dato.iloc[0])/sd*100:.1f} % de todos los nulos)')
print('  Lectura: n_oferentes no "falta". No existe, porque una compra por catalogo no tiene')
print('  proceso de ofertas. Dentro del universo de competencia la cobertura es practicamente')
print('  total. Lo que parecia un problema de calidad era una propiedad del procedimiento.')
nul.to_csv(f'{OUT}nulos_oferentes_por_clase.csv', index=False)

# ---------- Montos NaN ----------
print('\n' + '='*74 + '\n5.7  NaN en monto_tender (las reglas del Paso 2 no los veian)\n' + '='*74)
print(con.sql("""
SELECT clase, count(*) AS procesos, sum(CASE WHEN isnan(monto_tender) THEN 1 ELSE 0 END) AS nan,
       sum(CASE WHEN monto_tender IS NULL THEN 1 ELSE 0 END) AS nulos
FROM bm4t GROUP BY 1 HAVING nan > 0 OR nulos > 0 ORDER BY 1""").df().to_string(index=False))
print('NaN no es cero ni nulo: cualquier suma o promedio que lo incluya devuelve NaN y')
print('contamina en silencio. Hay que decidir si se convierten a nulo y declararlo.')

# ---------- Las cifras de competencia, bien calculadas ----------
print('\n' + '='*74 + '\n5.3b  Las cifras de competencia por universo\n' + '='*74)
comp = con.sql("""
SELECT '1 todo junto (diluido)' AS universo, count(*) AS procesos,
  round(sum(CASE WHEN isnan(monto_tender) THEN NULL ELSE monto_tender END)/1e6,1) AS monto_mm,
  round(avg(CASE WHEN proveedor_unico THEN 1.0 ELSE 0.0 END)*100,1) AS pct_of_unico FROM bm4t
UNION ALL SELECT '2 sin catalogo electronico', count(*),
  round(sum(CASE WHEN isnan(monto_tender) THEN NULL ELSE monto_tender END)/1e6,1),
  round(avg(CASE WHEN proveedor_unico THEN 1.0 ELSE 0.0 END)*100,1) FROM bm4t WHERE NOT es_catalogo
UNION ALL SELECT '3 sin catalogo ni regimen especial', count(*),
  round(sum(CASE WHEN isnan(monto_tender) THEN NULL ELSE monto_tender END)/1e6,1),
  round(avg(CASE WHEN proveedor_unico THEN 1.0 ELSE 0.0 END)*100,1) FROM bm4t
  WHERE NOT es_catalogo AND fam <> 'Regimen especial'
UNION ALL SELECT '4 solo abierto competitivo', count(*),
  round(sum(CASE WHEN isnan(monto_tender) THEN NULL ELSE monto_tender END)/1e6,1),
  round(avg(CASE WHEN proveedor_unico THEN 1.0 ELSE 0.0 END)*100,1) FROM bm4t WHERE universo_competencia
ORDER BY 1""").df()
print(comp.to_string(index=False))
print('\nLa fila 4 es la que va en la tesis. La 1 esta diluida por el catalogo y la 2 esta')
print('inflada por el regimen especial, donde el proveedor unico es legal.')
comp.to_csv(f'{OUT}universo_competencia.csv', index=False)

con.sql(f"""COPY (SELECT ocid, fam AS familia_procedimiento, clase, es_catalogo, universo_competencia,
                         tipo_contratacion AS tipo_original FROM bm4t)
            TO '{R}/c9_procedimiento_tmp.parquet' (FORMAT parquet, COMPRESSION zstd)""")
os.replace(f'{R}/c9_procedimiento_tmp.parquet', f'{OUT}procedimiento.parquet')
print(f'\n  -> {OUT}procedimiento.parquet, {OUT}taxonomia_procedimiento.csv,')
print(f'     {OUT}universo_competencia.csv, {OUT}nulos_oferentes_por_clase.csv')

In [ ]:
# ============ CUADERNO 9 · PASO 6 · El monto canonico ============
# Hallazgo del 5.7: monto_tender esta NULO en 216.469 registros, y 216.469 es exactamente el
# total de Subasta Inversa Electronica. O sea: el procedimiento competitivo mas importante del
# sistema NO tiene monto en esa columna. Tiene sentido — en una subasta inversa el presupuesto
# es referencial y el precio sale de la puja — pero significa que hay otra columna de monto y
# que hay que decidir cual es la buena, documentarlo y revisar que use el modelo.
# Requiere la vista bm4t del Paso 5b.

# ---------- 6.1 Todas las columnas que parecen monto ----------
desc = con.sql('DESCRIBE bm4').df()
CAND = desc[desc.column_name.str.contains('monto|valor|presupuesto|precio', case=False) &
            desc.column_type.isin(['DOUBLE', 'FLOAT', 'DECIMAL', 'BIGINT', 'INTEGER'])]
cols_monto = CAND.column_name.tolist()
print(f'--- Columnas candidatas a monto ({len(cols_monto)}) ---')
print(CAND.to_string(index=False))

# ---------- 6.2 Cobertura de cada una, por clase de procedimiento ----------
print('\n' + '='*74 + '\n6.2  Cobertura de cada columna de monto, por clase\n' + '='*74)
sel = ', '.join([f"round(avg(CASE WHEN {c} IS NOT NULL THEN 1.0 ELSE 0.0 END)*100, 1) AS \"{c}\""
                 for c in cols_monto])
cob = con.sql(f'SELECT clase, count(*) AS procesos, {sel} FROM bm4t GROUP BY 1 ORDER BY 1').df()
print(cob.to_string(index=False))
print('\nSe lee asi: el numero es el porcentaje de registros donde esa columna tiene dato.')
print('Una columna util para la tesis tiene cobertura alta en TODAS las clases, o al menos')
print('en el universo de competencia.')
cob.to_csv(f'{OUT}cobertura_montos.csv', index=False)

print('\n--- Suma de cada columna, en millones ---')
sel2 = ', '.join([f'round(sum({c})/1e6, 1) AS "{c}"' for c in cols_monto])
print(con.sql(f'SELECT {sel2} FROM bm4').df().T.rename(columns={0: 'millones'}).to_string())

# ---------- 6.3 Cual columna uso el filtro de procesos_sin_monto ----------
print('\n' + '='*74 + '\n6.3  Que columna decidio el filtro de los 16.807 sin monto\n' + '='*74)
sm = f'{R}/procesos_sin_monto.parquet'
if os.path.exists(sm):
    cs = {r[0] for r in con.sql(f"DESCRIBE SELECT * FROM read_parquet('{sm}')").fetchall()}
    comunes = [c for c in cols_monto if c in cs]
    if comunes:
        q = ', '.join([f"sum(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS \"{c}\"" for c in comunes])
        r = con.sql(f"SELECT count(*) AS filas, {q} FROM read_parquet('{sm}')").df()
        print(r.T.rename(columns={0: 'nulos'}).to_string())
        print('\nLa columna donde los 16.807 estan TODOS nulos es la que se uso para filtrar.')
        print('Hay que escribirlo asi en la cadena de filtros, nombrando la columna.')

# ---------- 6.4 Propuesta de monto canonico ----------
# Orden: primero el monto efectivamente adjudicado o contratado; si no hay, el referencial.
# Se declara el orden y se guarda de que columna salio cada fila, para poder auditarlo.
ORDEN = [c for c in ['monto_adjudicado', 'monto_contrato', 'monto_final', 'monto_award',
                     'valor_adjudicado', 'monto_tender', 'presupuesto_referencial']
         if c in cols_monto] + [c for c in cols_monto if c not in (
         'monto_adjudicado','monto_contrato','monto_final','monto_award','valor_adjudicado',
         'monto_tender','presupuesto_referencial')]
print('\n' + '='*74 + f'\n6.4  Monto canonico por orden de preferencia\n' + '='*74)
print('  Orden propuesto:', ' -> '.join(ORDEN))
coal = 'COALESCE(' + ', '.join([f'CASE WHEN {c} IS NOT NULL AND NOT isnan({c}) THEN {c} END'
                                if True else c for c in ORDEN]) + ')'
fuente = 'CASE ' + ' '.join([f"WHEN {c} IS NOT NULL AND NOT isnan({c}) THEN '{c}'" for c in ORDEN]) + " ELSE 'sin monto' END"
con.sql(f"""CREATE OR REPLACE VIEW bm4m AS
            SELECT ocid, clase, fam, {coal} AS monto_canonico, {fuente} AS monto_origen FROM bm4t""")
print('\n--- De que columna sale el monto, por clase ---')
print(con.sql("""SELECT clase, monto_origen, count(*) AS procesos,
                        round(sum(monto_canonico)/1e6, 1) AS millones
                 FROM bm4m GROUP BY 1,2 ORDER BY 1, 3 DESC""").df().to_string(index=False))
tot = con.sql("""SELECT count(*) AS procesos,
                        sum(CASE WHEN monto_origen = 'sin monto' THEN 1 ELSE 0 END) AS sin_monto,
                        round(sum(monto_canonico)/1e6, 1) AS millones FROM bm4m""").df()
print('\n--- Total con el monto canonico ---')
print(tot.to_string(index=False))
print('\nCompara ese total contra la suma de monto_tender del 6.2. La diferencia es el dinero')
print('que hoy NO esta entrando en ninguna cifra agregada de la tesis.')

con.sql(f"""COPY (SELECT * FROM bm4m) TO '{R}/c9_monto_tmp.parquet' (FORMAT parquet, COMPRESSION zstd)""")
os.replace(f'{R}/c9_monto_tmp.parquet', f'{OUT}monto_canonico.parquet')
print(f'\n  -> {OUT}monto_canonico.parquet (ocid, monto_canonico, monto_origen)')
print(f'  -> {OUT}cobertura_montos.csv')

# ---------- 6.5 Que columna usa el panel del modelo ----------
print('\n' + '='*74 + '\n6.5  Que monto usa hoy el modelo\n' + '='*74)
pc = {r[0] for r in con.sql(f"DESCRIBE SELECT * FROM read_parquet('{R}/c8_panel.parquet')").fetchall()}
usa = [c for c in pc if any(k in c.lower() for k in ('monto', 'valor', 'precio'))]
print('  Columnas con monto en el panel:', sorted(usa)[:20])
print('\n  Si el panel se construyo sumando monto_tender, entonces las variables de monto del')
print('  modelo estan vacias para las 216.469 subastas inversas, y el percentil de 0,858 que')
print('  sale en la figura del perfil de red hay que recalcularlo con el monto canonico.')
print('  Esa es la verificacion que toca hacer antes de dar la figura por buena.')

In [ ]:
# ============ CUADERNO 9 · PASO 7 · Corregir el monto y medir el dano ============
# Lo que el Paso 6 establecio:
#   monto_final  -> 100 % de cobertura en TODAS las clases. Total 66.570,9 millones.
#   monto_tender -> solo 15,3 % en el universo abierto competitivo, porque esta nulo en las
#                   216.469 subastas inversas. Total 51.327,3 millones.
# Y hay una consecuencia que hay que medir: en la auditoria de ausencias (14.1), la variable
# cuya AUSENCIA mas discriminaba era monto_tender, con 0,612 de 107 variables. Ahora sabemos
# por que: su ausencia identifica "es una subasta inversa". El modelo tenia un canal lateral
# hacia el tipo de procedimiento a traves de un patron de nulos.
# Requiere las vistas bm4 y bm4t del Paso 5b.

# ---------- 7.1 Cual columna uso el panel ----------
print('='*74 + '\n7.1  De que columna salieron las variables de monto del panel\n' + '='*74)
con.sql(f"""CREATE OR REPLACE VIEW agg AS
SELECT proveedor_ruc, anio_proceso,
       sum(monto_tender) AS t_tender, sum(monto_award) AS t_award, sum(monto_final) AS t_final,
       max(monto_final) AS mx_final
FROM bm4 WHERE TRIM(COALESCE(proveedor_ruc,'')) <> '' GROUP BY 1,2""")
import pyarrow.parquet as pq
COLS_PANEL = set(pq.ParquetFile(f'{R}/c8_panel.parquet').schema.names)
pedir = [c for c in ['proveedor_ruc','anio_proceso','monto_anio','monto_max_anio'] if c in COLS_PANEL]
faltan = [c for c in ['monto_anio','monto_max_anio'] if c not in COLS_PANEL]
if faltan: print('  aviso: el panel no tiene', faltan)
P = pd.read_parquet(f'{R}/c8_panel.parquet', columns=pedir)
A = con.sql('SELECT * FROM agg').df()
M = P.merge(A, on=['proveedor_ruc', 'anio_proceso'], how='left')
print(f'  filas comparadas: {len(M):,}')
for c in (['t_tender', 't_award', 't_final'] if 'monto_anio' in M.columns else []):
    ok = M[['monto_anio', c]].dropna()
    r = ok.monto_anio.corr(ok[c]) if len(ok) > 10 else np.nan
    igual = float(np.isclose(ok.monto_anio, ok[c], rtol=1e-6).mean()) if len(ok) else np.nan
    print(f'  monto_anio vs suma de {c[2:]:8s}: correlacion {r:.4f} | coincide exacto {igual*100:5.1f} %')
if 'monto_max_anio' in M.columns:
    ok = M[['monto_max_anio', 'mx_final']].dropna()
    print(f'  monto_max_anio vs max de final: coincide exacto '
          f'{np.isclose(ok.monto_max_anio, ok.mx_final, rtol=1e-6).mean()*100:5.1f} %')
print('\n  La columna con coincidencia cercana al 100 % es la que se uso. Si es tender, hay que')
print('  reconstruir todas las variables de monto del panel con final.')

# ---------- 7.2 Agregados correctos, para reemplazar ----------
con.sql(f"""COPY (SELECT proveedor_ruc, anio_proceso,
                   sum(monto_final) AS monto_anio_final,
                   max(monto_final) AS monto_max_anio_final,
                   count(*) AS n_procesos_anio,
                   sum(CASE WHEN universo_competencia THEN monto_final ELSE 0 END) AS monto_anio_competitivo,
                   sum(CASE WHEN es_catalogo THEN monto_final ELSE 0 END) AS monto_anio_catalogo
            FROM bm4t WHERE TRIM(COALESCE(proveedor_ruc,'')) <> '' GROUP BY 1,2)
            TO '{R}/c9_monto_panel_tmp.parquet' (FORMAT parquet, COMPRESSION zstd)""")
os.replace(f'{R}/c9_monto_panel_tmp.parquet', f'{OUT}monto_panel.parquet')
print(f'\n  -> {OUT}monto_panel.parquet (para reemplazar monto_anio y monto_max_anio)')

# ---------- 7.3 La fila de monto de la figura 2, recalculada ----------
print('\n' + '='*74 + '\n7.3  El percentil de monto de las empresas del catastro\n' + '='*74)
base = ['proveedor_ruc', 'anio_proceso', 'y_fantasma']
act  = next((c for c in ['n_contracts', 'n_procesos', 'red_n_contratos'] if c in COLS_PANEL), None)
pan = pd.read_parquet(f'{R}/c8_panel.parquet',
                      columns=base + [c for c in ['monto_tender', act] if c and c in COLS_PANEL])
mp = pd.read_parquet(f'{OUT}monto_panel.parquet')
Q = pan.merge(mp, on=['proveedor_ruc', 'anio_proceso'], how='left')
Q['decil'] = Q.groupby('anio_proceso')[act].transform(
    lambda s: pd.qcut(s.rank(method='first'), 10, labels=False, duplicates='drop'))
filas = []
for col, etq in [('monto_tender', 'monto_tender (como esta en la figura)'),
                 ('monto_anio_final', 'monto_final (cobertura completa)'),
                 ('monto_anio_competitivo', 'monto solo del universo competitivo')]:
    if col not in Q.columns: continue
    pct = Q.groupby(['anio_proceso', 'decil'])[col].rank(pct=True)
    filas.append((etq, round(float(np.nanmedian(pct[Q.y_fantasma == 1])), 3),
                  round(float(pd.isna(Q[col]).mean()) * 100, 1)))
t73 = pd.DataFrame(filas, columns=['variable', 'percentil_fantasmas', 'pct_nulos'])
print(t73.to_string(index=False))
print('\n  Si el percentil cambia mucho entre la primera y la segunda fila, la figura 2 hay que')
print('  corregirla antes de ponerla en la tesis, y el rotulo tambien: monto_tender es el monto')
print('  referencial del pliego, NO el monto adjudicado.')
t73.to_csv(f'{OUT}percentil_monto_corregido.csv', index=False)

# ---------- 7.4 La concentracion del gasto, con el monto bueno ----------
print('\n' + '='*74 + '\n7.4  Concentracion del gasto\n' + '='*74)
for col, etq in [('monto_tender', 'monto_tender'), ('monto_final', 'monto_final')]:
    d = con.sql(f"""
    WITH p AS (SELECT proveedor_ruc, sum({col}) AS m FROM bm4
               WHERE TRIM(COALESCE(proveedor_ruc,'')) <> '' AND {col} IS NOT NULL
               GROUP BY 1 HAVING m > 0),
         r AS (SELECT m, row_number() OVER (ORDER BY m DESC) AS pos, count(*) OVER () AS n,
                      sum(m) OVER () AS tot FROM p)
    SELECT round(sum(CASE WHEN pos <= n*0.01 THEN m ELSE 0 END)/max(tot)*100, 1) AS top1,
           round(sum(CASE WHEN pos <= n*0.10 THEN m ELSE 0 END)/max(tot)*100, 1) AS top10,
           round(sum(CASE WHEN pos >  n*0.50 THEN m ELSE 0 END)/max(tot)*100, 1) AS mitad_inferior,
           count(*) AS proveedores, round(max(tot)/1e6, 1) AS millones FROM r""").df()
    d.insert(0, 'columna', etq)
    print(d.to_string(index=False))
print('\n  Las cifras de concentracion de la tesis se calcularon con una de las dos. Hay que')
print('  reportar las de monto_final y decir que se corrigio.')

# ---------- 7.5 El alcance del problema ----------
print('\n' + '='*74 + '\n7.5  Variables del panel que hay que revisar\n' + '='*74)
pc = [r[0] for r in con.sql(f"DESCRIBE SELECT * FROM read_parquet('{R}/c8_panel.parquet')").fetchall()]
sosp = [c for c in pc if 'tender' in c.lower()]
print('  Derivadas de monto_tender (fuera del modelo, o reconstruidas con monto_final):')
print('   ', sosp if sosp else 'ninguna')
print('\n  Decision a tomar y escribir:')
print('   1. monto_final es el monto canonico del trabajo. Cobertura 100 %, total 66.570,9 M.')
print('   2. monto_tender sale del conjunto de variables: su patron de nulos identifica el tipo')
print('      de procedimiento, y eso es un atajo que el modelo no debe tener.')
print('   3. Los 16.807 excluidos por falta de monto estaban nulos en las TRES columnas, asi')
print('      que ese filtro esta bien y se puede defender nombrando las tres.')
print(f'\n  -> {OUT}monto_panel.parquet, {OUT}percentil_monto_corregido.csv')

In [ ]:
# ============ CORRECCION DEL MONTO · dos cambios de una linea ============
# El Paso 7 establecio que monto_anio del panel ya venia de monto_final (coincidencia 100 %).
# Entonces el dano se limita a dos cosas, y las dos se arreglan sin reconstruir nada.

# ---------- 1. monto_tender sale del conjunto de variables ----------
# Motivo: su patron de nulos identifica "es una subasta inversa". Es un confundidor con el tipo
# de procedimiento, no una variable de conducta. Se excluye por ORIGEN, igual que las del SRI.
with open(f'{R}/c8_vars.json', encoding='utf-8') as f: VARS = json.load(f)
FUERA = {'monto_tender'}
antes = {k: len(v) for k, v in VARS.items()}
VARS = {k: [c for c in v if c not in FUERA] for k, v in VARS.items()}
with open(f'{R}/c8_vars.json', 'w', encoding='utf-8') as f:
    json.dump(VARS, f, ensure_ascii=False, indent=1)
print('--- Conjuntos de variables ---')
for k in VARS:
    print(f'  {k:18s} {antes[k]:3d} -> {len(VARS[k]):3d}' + ('  (salio monto_tender)' if antes[k] != len(VARS[k]) else ''))
print('\n  c8_vars.json actualizado. El modelo hay que volver a correrlo (Pasos 7 a 14 del')
print('  cuaderno 8), pero es una sola variable de 142: el resultado no deberia moverse mucho.')
print('  Y si se mueve, eso mismo es lo que habia que saber.')

# ---------- 2. La figura 2 usa monto_anio, no monto_tender ----------
# En celda_15_figuras.py, en la lista PERFIL, cambia esta linea:
print("""
--- En celda_15_figuras.py, dentro de PERFIL, reemplaza esta linea ---

    ('monto_anio','Monto contratado en el año'),
por esta:

    ('monto_anio','Monto contratado en el anio'),

Dos motivos: monto_anio sale de monto_final y tiene cobertura completa, y el rotulo anterior
era incorrecto porque monto_tender es el monto REFERENCIAL del pliego, no el adjudicado.
La fila pasa de 0,858 a 0,788. La conclusion no cambia: las empresas del catastro siguen
contratando montos muy por encima de proveedores comparables.
""")

# ---------- 3. El dato extra que vale la pena poner en el texto ----------
print('--- Lo que conviene escribir junto a la figura ---')
print('  monto_tender (cobertura 77,6 %) ........ 0,858   <- lo que decia la version anterior')
print('  monto contratado (cobertura 100 %) ..... 0,788   <- la cifra correcta')
print('  monto del universo competitivo ......... 0,871   <- restringido a procesos con puja')
print('\n  La tercera es la mas interesante y vale una frase en la tesis: cuando se mira solo')
print('  donde hubo competencia real, el monto de estas empresas se destaca MAS, no menos.')